# Notebook d'Inférence - Prédictions Consommation Électrique

Ce notebook permet de faire des prédictions de consommation électrique pour la journée suivante.

## Fonctionnalités :
1. Chargement du modèle depuis MLflow
2. Génération des features pour la journée suivante (toutes les 30 minutes)
3. Prédictions sur toutes les plages horaires
4. Stockage des prédictions en base de données

## Structure de la table de stockage :
- `prediction_id`: TEXT (UUID)
- `prediction_timestamp`: TIMESTAMP
- `prediction_date`: TIMESTAMP
- `prediction_index`: INTEGER
- `prediction`: DOUBLE PRECISION
- `confidence`: DOUBLE PRECISION
- `model_version`: TEXT
- `created_at`: TIMESTAMP

## 0. Initialisation et Configuration

In [1]:
import os
import logging
import warnings
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Ignorer certains warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

print("✅ Initialisation terminée")

✅ Initialisation terminée


## 1. Configuration de la Base de Données

In [2]:
# Configuration PostgreSQL (à adapter selon votre environnement)
DB_URI = os.getenv("PREDICTIONS_POSTGRES_URI")

PREDICTIONS_TABLE = os.getenv("PREDICTIONS_TABLE", "predictions")

if DB_URI:
    print("📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI")
    print(f"  Table: {PREDICTIONS_TABLE}")
else:
    DB_CONFIG = {
        "host": os.getenv("DB_HOST", "localhost"),
        "port": os.getenv("DB_PORT", "5432"),
        "database": os.getenv("DB_NAME", "jinsudai"),
        "user": os.getenv("DB_USER", "postgres"),
        "password": os.getenv("DB_PASSWORD", ""),
    }
    print(f"📊 Configuration DB :")
    print(f"  Host: {DB_CONFIG['host']}")
    print(f"  Port: {DB_CONFIG['port']}")
    print(f"  Database: {DB_CONFIG['database']}")
    print(f"  Table: {PREDICTIONS_TABLE}")

📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI
  Table: predictions


## 2. Configuration MLflow

In [3]:
## 2. Configuration MLflow
import sys
sys.path.insert(0, str(Path.cwd() / "src"))

from ml.utils.pipelines import PredictionPipeline

# Configuration MLflow
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://jinsudai-mlflow.hf.space/")
MODEL_NAME = os.getenv("MODEL_NAME", "consumption_model_test")
MODEL_ALIAS = os.getenv("MODEL_ALIAS", "prod")  # 'prod' ou 'staging'

pipeline = PredictionPipeline(
    mlflow_uri=MLFLOW_TRACKING_URI,
    experiment_name=None,
    db_uri=DB_URI
)

pipeline_ready = pipeline.setup()

print("🔧 Pipeline d'inférence initialisé")
print(f"  Modèle: {MODEL_NAME}")
print(f"  Alias: {MODEL_ALIAS}")
print(f"  Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"  Pipeline prêt: {pipeline_ready}")


c:\Users\SustCoop\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Local\pypoetry\Cache\virtualenvs\ml-6PV5fMfH-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-12 06:10:50,352 - ml.utils.pipelines.Prediction_pipeline - INFO - Pipeline d'inférence initialisé
2026-06-12 06:10:50,353 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 1: CONFIGURATION ===
2026-06-12 06:10:53,421 - ml.utils.models.models_inference - INFO - MLflow connecté à https://jinsudai-mlflow.hf.space/
2026-06-12 06:10:53,423 - ml.utils.pipelines.Prediction_pipeline - INFO - MLflow configuré
2026-06-12 06:10:53,651 - ml.utils.pipelines.database_handler - INFO - Connexion à la base de données vérifiée
2026-06-12 06:10:53,965 - ml.utils.pipelines.database_handler - INFO - Table predictions_

🔧 Pipeline d'inférence initialisé
  Modèle: consumption_model_test
  Alias: prod
  Tracking URI: https://jinsudai-mlflow.hf.space/
  Pipeline prêt: True


## 3. Chargement du Modèle depuis MLflow

In [4]:
print(f"🔄 Chargement du modèle {MODEL_NAME} (alias: {MODEL_ALIAS})...")

try:
    if not pipeline.load_model(MODEL_NAME, alias_prod=MODEL_ALIAS):
        raise RuntimeError("Échec du chargement du modèle via le pipeline")

    model_info = pipeline.get_model_info() or {}
    model_version = model_info.get("version", "unknown")

    print(f"✅ Modèle chargé avec succès")
    print(f"  Version: {model_version}")

except Exception as e:
    logger.error(f"Erreur lors du chargement du modèle: {e}")
    raise

2026-06-12 06:10:53,999 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 2: CHARGEMENT DU MODÈLE ===


🔄 Chargement du modèle consumption_model_test (alias: prod)...


2026-06-12 06:10:55,023 - ml.utils.models.models_inference - INFO - Model registry source: runs:/e5c4f5a023914c958cdc3829bcf64e15/model
2026-06-12 06:10:59,392 - botocore.credentials - INFO - Found credentials in environment variables.
2026/06/12 06:11:03 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at s3://mlflow-artifacts/2/e5c4f5a023914c958cdc3829bcf64e15/artifacts/model. Returning destination path.
2026-06-12 06:11:26,730 - ml.utils.models.models_inference - WARNING - Chargement sklearn échoué (Could not find a registered artifact repository for: c:\Users\SustCoop\AppData\Local\Temp\tmpqlocevqd. Currently registered schemes are: ['', 'file', 's3', 'r2', 'b2', 'gs', 'wasbs', 'ftp', 'sftp', 'dbfs', 'hdfs', 'viewfs', 'runs', 'models', 'http', 'https', 'mlflow-artifacts', 'abfss']), tentative chargement pyfunc pour models:/consumption_model_test@prod
2026/06/12 06:11:30 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at s3://mlflow-a

✅ Modèle chargé avec succès
  Version: 13


## 4. Génération des Features pour la Journée Suivante

In [5]:

features_df = pipeline.generate_data()


2026-06-12 06:12:15,561 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 3: GÉNÉRATION DES DONNÉES ===
2026-06-12 06:12:20,338 - ml.connectors.holidays.holidays_api - INFO - Données vacances scolaires chargées: 570 périodes


AttributeError: Can only use .dt accessor with datetimelike values

## 5. Préparation des Features pour le Modèle

In [ ]:

# Préparer les features via PredictionPipeline
features_prep = pipeline.prepare_features()

print(f"✅ Features préparées via PredictionPipeline: {features_prep.shape}")
display(features_prep.head())

2026-06-12 03:57:00,700 - ml.utils.data.data_transformer - INFO - Aucune colonne valide à supprimer après vérification
2026-06-12 03:57:00,704 - ml.utils.data.data_transformer - INFO - Nettoyage des données terminé


✅ Features préparées via PredictionPipeline: (72, 7)


,Horodate,temperature_2m_mean,relative_humidity_mean,precipitation_sum,is_vacances,jour de la semaine,jour férié
0,2026-06-12 00:00:00,22.483571,74.693523,0.193405,0,Friday,0
1,2026-06-12 01:00:00,19.308678,62.220802,0.103216,0,Friday,0
2,2026-06-12 02:00:00,23.238443,63.522977,0.020815,1,Friday,0
3,2026-06-12 03:00:00,27.615149,59.239346,0.446889,1,Friday,0
4,2026-06-12 04:00:00,18.829233,41.143861,0.565926,1,Friday,0


## 6. Prédictions

In [ ]:
print("🔄 Prédictions en cours...")

try:
    if not pipeline_ready:
        raise RuntimeError("Le pipeline d'inférence n'est pas prêt")

    prediction_ok = pipeline.run_predictions()
    if not prediction_ok:
        raise RuntimeError("Échec des prédictions via le pipeline")

    predictions_df = pipeline.get_predictions_df()
    if predictions_df is None or predictions_df.empty:
        raise RuntimeError("Aucune prédiction générée")

    print(f"✅ Prédictions terminées: {len(predictions_df)} valeurs")
    print()
    print("📊 Statistiques des prédictions:")
    print(f"  Min: {predictions_df['prediction'].min():.2f}")
    print(f"  Max: {predictions_df['prediction'].max():.2f}")
    print(f"  Mean: {predictions_df['prediction'].mean():.2f}")
    print(f"  Std: {predictions_df['prediction'].std():.2f}")

    if "confidence" in predictions_df.columns:
        print()
        print(f"📈 Scores de confiance disponibles: {predictions_df['confidence'].shape}")

    display(predictions_df.head())
except Exception as e:
    logger.error(f"Erreur lors des prédictions: {e}")
    raise


2026-06-12 03:57:00,768 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 4: EXÉCUTION DES PRÉDICTIONS ===


🔄 Prédictions en cours...


2026-06-12 03:57:01,973 - ml.utils.models.models_inference - INFO - Prédictions générées pour 72 échantillons
2026-06-12 03:57:01,975 - ml.utils.data.data_prediction - INFO - Prédictions ajoutées au DataFrame: (72, 8)
2026-06-12 03:57:01,977 - ml.utils.pipelines.Prediction_pipeline - INFO - Prédictions générées: 72 échantillons


✅ Prédictions terminées: 72 valeurs

📊 Statistiques des prédictions:
  Min: 509.65
  Max: 826.90
  Mean: 655.81
  Std: 76.63


,Horodate,temperature_2m_mean,relative_humidity_mean,precipitation_sum,is_vacances,jour de la semaine,jour férié,prediction
0,2026-06-12 00:00:00,22.483571,74.693523,0.193405,0,Friday,0,530.227844
1,2026-06-12 01:00:00,19.308678,62.220802,0.103216,0,Friday,0,588.431091
2,2026-06-12 02:00:00,23.238443,63.522977,0.020815,1,Friday,0,598.352295
3,2026-06-12 03:00:00,27.615149,59.239346,0.446889,1,Friday,0,661.246277
4,2026-06-12 04:00:00,18.829233,41.143861,0.565926,1,Friday,0,768.778442


## 7. Préparation des Données pour Stockage

In [ ]:
predictions_df = pipeline.get_predictions_df()

if predictions_df is None or predictions_df.empty:
    raise RuntimeError("Aucune prédiction disponible pour le stockage")

print(f"✅ Données prêtes pour stockage: {len(predictions_df)} enregistrements")
display(predictions_df.head())

✅ Données prêtes pour stockage: 72 enregistrements


,Horodate,temperature_2m_mean,relative_humidity_mean,precipitation_sum,is_vacances,jour de la semaine,jour férié,prediction
0,2026-06-12 00:00:00,22.483571,74.693523,0.193405,0,Friday,0,530.227844
1,2026-06-12 01:00:00,19.308678,62.220802,0.103216,0,Friday,0,588.431091
2,2026-06-12 02:00:00,23.238443,63.522977,0.020815,1,Friday,0,598.352295
3,2026-06-12 03:00:00,27.615149,59.239346,0.446889,1,Friday,0,661.246277
4,2026-06-12 04:00:00,18.829233,41.143861,0.565926,1,Friday,0,768.778442


## 8. Connexion à la Base de Données

In [ ]:
if DB_URI:
    print("📌 Le pipeline stockera les prédictions dans la base de données via PredictionPipeline.")
else:
    print("⚠️ Aucune URI DB fournie : le stockage via PredictionPipeline sera ignoré.")

📌 Le pipeline stockera les prédictions dans la base de données via PredictionPipeline.


## 9. Utilisation du Flow Prefect (Optionnel)

In [ ]:
# Optionnel : Utiliser le flow Prefect pour l'exécution
# from ml.workflows.prediction_flow import prediction_full_pipeline
# 
# result = prediction_full_pipeline(
#     model_name=MODEL_NAME,
#     n_days=1,
#     n_samples_per_day=48,
#     alias_prod=MODEL_ALIAS
# )
# print(f"Résultat du flow: {result}")

print("💡 Pour utiliser le flow Prefect, décommentez le code ci-dessus")

In [ ]:
print("🔄 Stockage des prédictions en cours via PredictionPipeline...")

try:
    if not pipeline_ready:
        raise RuntimeError("Le pipeline d'inférence n'est pas prêt")

    stored = pipeline.store_predictions()
    if not stored:
        raise RuntimeError("Échec du stockage des prédictions via le pipeline")

    print("✅ Prédictions stockées via PredictionPipeline")
except Exception as e:
    logger.error(f"Erreur lors du stockage des prédictions: {e}")
    raise


2026-06-12 03:57:02,118 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 5: STOCKAGE EN BASE DE DONNÉES ===


🔄 Stockage des prédictions en cours via PredictionPipeline...


2026-06-12 03:57:02,229 - ml.utils.pipelines.Prediction_pipeline - WARNING - Les prédictions ne sont pas toutes espacées de 30 minutes. Écarts détectés: <TimedeltaArray>
['0 days 01:00:00']
Length: 1, dtype: timedelta64[ns]
2026-06-12 03:57:02,241 - ml.utils.pipelines.Prediction_pipeline - INFO - Stockage des prédictions               Horodate  temperature_2m_mean  relative_humidity_mean  precipitation_sum  is_vacances jour de la semaine  jour férié  prediction prediction_timestamp     prediction_date  prediction_index
71 2026-06-14 23:00:00            27.690183               77.588612           0.012308            0             Sunday           0  620.077393  2026-06-14 23:00:00 2026-06-14 23:00:00                72
70 2026-06-14 22:00:00            21.806978               44.063040           0.164789            0             Sunday           0  509.652466  2026-06-14 22:00:00 2026-06-14 22:00:00                71
69 2026-06-14 21:00:00            16.774401               64.109861    

✅ Prédictions stockées via PredictionPipeline
